NoteBook 04

This notebook implements a hybrid recommendation model that combines semantic similarity (SBERT) with structured features (e.g., experience, skills score).

🔹 Key Steps:
    1.Structured Features
    2.Hybrid Score
    3.Recommend

🔹 Final Formula:
    Hybrid Score = 0.8 × Text Similarity + 0.2 × Structured Similarity
    
🔹 Purpose:
    Achieve better accuracy and balance
    Combine strengths of both text and structured data

1. Structured Features

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# 🔹 Load data
resume_df = pd.read_csv('../data/processed/processed_resume.csv')
jobs_df   = pd.read_csv('../data/processed/processed_jobs.csv')

# 🔹 Load embeddings (IMPORTANT)
resume_emb = np.load('../models/resume_emb.npy')
job_emb    = np.load('../models/job_emb.npy')

# 🔹 Structured features
resume_struct = resume_df[
    ['cgpa','experience_years','skills_score','soft_skills_score']
].fillna(0)

scaler = StandardScaler()
resume_struct_scaled = scaler.fit_transform(resume_struct)

# dummy job structure
jobs_struct_scaled = np.zeros((jobs_df.shape[0], resume_struct_scaled.shape[1]))

# 🔹 Hybrid similarity
text_sim   = cosine_similarity(resume_emb, job_emb)
struct_sim = cosine_similarity(resume_struct_scaled, jobs_struct_scaled)

final_sim = 0.8 * text_sim + 0.2 * struct_sim

2. Hybrid Score

In [2]:
text_sim = cosine_similarity(resume_emb, job_emb)
struct_sim = cosine_similarity(resume_struct_scaled, jobs_struct_scaled)

final_sim = 0.8 * text_sim + 0.2 * struct_sim

np.save('../models/final_sim.npy', final_sim)

3. Recommend

In [3]:
def recommend(idx, top_n=5):
    scores = final_sim[idx]
    top_idx = scores.argsort()[::-1][:top_n]
    return jobs_df.iloc[top_idx][['title','location']]